# Task 6: Thí nghiệm A/B character consistency (Kaggle)

Notebook sinh các cặp truyện A (prompt độc lập) và B (shared character descriptions) cho nhiều storyboard bằng
`SDXLImageGenerator`, rồi tạo gói chấm ẩn nhãn hai tầng (`core`, `extra`).

**Yêu cầu khi chạy**
- Accelerator: GPU T4 (notebook chỉ dùng một GPU nhờ `CUDA_VISIBLE_DEVICES=0`). Internet: bật.
- Thêm vào notebook (Add data) bộ dữ liệu **private** chứa các file storyboard; tên file là `<story_id>.json`.
- `SOURCE_COMMIT` ở Cell 2 ghim commit của lần chạy chính thức 25/11/2026; muốn chạy lại với mã nguồn mới thì thay bằng commit khác (đủ 40 ký tự).
- Chạy từ đầu (Run All) trong phiên mới. Không commit notebook đã chạy (có output chứa dữ liệu private); xóa output trước khi commit.

**Kết quả** nằm trong `/kaggle/working`: `ab_experiment_full.zip` (đầy đủ, có khóa A/B, KHÔNG gửi người chấm),
`ab_for_raters_core.zip` và `ab_for_raters_extra.zip` (chỉ những file này gửi người chấm).

## 1. Kiểm tra môi trường

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # một T4; phải đặt trước khi import torch

import platform
import sys

import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
assert torch.cuda.is_available(), "Không có GPU: bật Accelerator GPU T4 trong Settings rồi chạy lại"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}, {p.total_memory / 2**30:.2f} GiB")

## 2. Lấy đúng mã nguồn (commit cố định)

In [ ]:
import importlib.metadata as md
import subprocess

REPO_URL = "https://github.com/trannam1084/comic-generation.git"
SOURCE_COMMIT = "3772bb4e947e70265c1ae0b46c039a71b1004555"  # commit của lần chạy chính thức 25/11/2026 (feature/image-pipeline)
WORKDIR = "/kaggle/working/comic-generation"

assert len(SOURCE_COMMIT) == 40 and all(c in "0123456789abcdef" for c in SOURCE_COMMIT), \
    "Điền SOURCE_COMMIT là mã commit đầy đủ 40 ký tự (git rev-parse HEAD)"

GIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT="0")  # không bao giờ chờ nhập mật khẩu


def git(*args):
    done = subprocess.run(["git", *args], capture_output=True, text=True, env=GIT_ENV)
    if done.returncode != 0:
        raise RuntimeError(f"git {args[0]} that bai (ma {done.returncode}): {done.stderr.strip()[:300]}")
    return done.stdout.strip()


if not os.path.isdir(WORKDIR):
    git("clone", "--no-checkout", REPO_URL, WORKDIR)
git("-C", WORKDIR, "checkout", "--detach", SOURCE_COMMIT)
head = git("-C", WORKDIR, "rev-parse", "HEAD")
assert head == SOURCE_COMMIT, f"HEAD {head} khac commit mong doi"
print("HEAD:", head)
print("Thu muc sach (khong co thay doi cuc bo):", git("-C", WORKDIR, "status", "--porcelain") == "")

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)

for package in ["pydantic", "Pillow", "PyYAML", "torch", "diffusers", "transformers", "huggingface_hub"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: KHONG CO")

## 3. Nạp và kiểm tra storyboard (CPU, trước khi nạp model)

In [ ]:
import json
from pathlib import Path

from src.image_generator import ab_experiment as ab
from src.image_generator.ab_rubric import rated_characters
from src.storyboard.schema import Storyboard

EXPECTED_STORY_IDS = ["story_013", "story_020_v2", "story_027", "story_033_v2", "story_050"]
BASE_SEEDS = (100, 200)  # seed đầu tiên là tầng lõi (core), seed sau là tầng bổ sung (extra)
SHUFFLE_SEED = 2026
INPUT_ROOT = Path("/kaggle/input")

found = {}
for path in sorted(INPUT_ROOT.rglob("*.json")):
    found.setdefault(path.stem, []).append(path)

missing = [s for s in EXPECTED_STORY_IDS if s not in found]
duplicated = {s: [str(p) for p in found[s]] for s in EXPECTED_STORY_IDS if s in found and len(found[s]) > 1}
if missing or duplicated:
    raise RuntimeError(
        f"Thiếu file storyboard: {missing}; trùng tên: {duplicated}. "
        f"Các tên tìm thấy trong {INPUT_ROOT}: {sorted(found)}"
    )

storyboards = {}
for story_id in EXPECTED_STORY_IDS:
    path = found[story_id][0]
    raw = json.loads(path.read_text(encoding="utf-8"))
    Storyboard.model_validate(raw)  # lỗi cấu trúc nổi lên ở đây, trước khi tốn thời gian nạp model
    storyboards[story_id] = raw
    rated = [c.character_id for c in rated_characters(raw)]
    print(f"{story_id}: {path} | nhân vật đủ điều kiện chấm nhất quán: {rated}")

specs = [ab.StorySpec(story_id, storyboards[story_id]) for story_id in EXPECTED_STORY_IDS]
ab.validate_stories(specs)
print(f"\nSố truyện: {len(specs)} | base_seeds: {BASE_SEEDS} | số truyện tranh sẽ sinh: {len(specs) * len(BASE_SEEDS) * 2}")

## 4. Nạp model và sinh các cặp A/B (cần GPU)

In [ ]:
import time

from src.image_generator.sdxl_generator import SDXLConfig, SDXLImageGenerator

OUT = Path("/kaggle/working/ab_experiment")
if OUT.exists():
    raise RuntimeError("Thư mục kết quả đã tồn tại: xóa hoặc đổi tên trước khi chạy để không ghi đè.")
RUN_DIR = OUT / "run"

generator = SDXLImageGenerator(SDXLConfig())  # mặc định giống notebook 03; model nạp ở lần sinh ảnh đầu tiên
torch.cuda.reset_peak_memory_stats()

start = time.perf_counter()
records = ab.run_ab_generation_stories(generator, stories=specs, base_seeds=BASE_SEEDS, out_dir=RUN_DIR)
total_sec = time.perf_counter() - start
peak_gib = torch.cuda.max_memory_allocated() / 2**30

print(f"Đã sinh {len(records)} truyện tranh trong {total_sec:.1f}s (gồm nạp model {generator.model_load_sec:.1f}s)")
print(f"VRAM đỉnh: {peak_gib:.2f} GiB")

## 5. Tạo gói chấm ẩn nhãn (core và extra)

In [ ]:
from src.image_generator import ab_pack as pack

PACKS = OUT / "packs"
keys = pack.build_blind_packs(
    ab.load_records(RUN_DIR), storyboards, BASE_SEEDS, PACKS, shuffle_seed=SHUFFLE_SEED
)
for tier, rows in keys.items():
    print(f"{tier}: {len(rows)} truyện tranh, mã {rows[0]['comic_id']} .. {rows[-1]['comic_id']}")

## 6. Lưu metadata, kiểm tra toàn vẹn và đóng gói

In [ ]:
import hashlib
import shutil


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


manifest = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
metadata = {
    "source_commit": SOURCE_COMMIT,
    "model": {"repo_id": generator.config.repo_id, "revision": generator.config.revision},
    "accelerator": torch.cuda.get_device_name(0),
    "base_seeds": list(BASE_SEEDS),
    "shuffle_seed": SHUFFLE_SEED,
    "stories": manifest["stories"],
    "n_comics": len(records),
    "model_load_sec": round(generator.model_load_sec, 2),
    "generation_total_sec": round(total_sec, 2),
    "peak_vram_gib": round(peak_gib, 2),
    "style": manifest["style"],
    "negative_prompt": manifest["negative_prompt"],
    "inference_params": manifest["inference_params"],
}
(OUT / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

lines = [f"{sha256(p)}  {p.relative_to(OUT).as_posix()}" for p in sorted(OUT.rglob("*")) if p.is_file()]
(OUT / "SHA256SUMS").write_text("\n".join(lines) + "\n", encoding="utf-8")

WORKING = Path("/kaggle/working")
archives = {"full (CÓ khóa A/B, không gửi người chấm)": shutil.make_archive(str(WORKING / "ab_experiment_full"), "zip", OUT)}
for tier in keys:
    archives[f"for_raters {tier} (gửi người chấm)"] = shutil.make_archive(
        str(WORKING / f"ab_for_raters_{tier}"), "zip", PACKS / tier / "for_raters"
    )
for label, archive in archives.items():
    print(f"{label}: {archive} | {Path(archive).stat().st_size / 2**20:.1f} MiB | sha256 {sha256(archive)}")
print(json.dumps({k: metadata[k] for k in ("n_comics", "model_load_sec", "generation_total_sec", "peak_vram_gib")}, indent=2))

## 7. Xem nhanh một cặp (chỉ để kiểm tra bằng mắt; bạn biết nhãn A/B nên không dùng để chấm)

In [ ]:
from IPython.display import display
from PIL import Image

pair = [r for r in records if r.story_id == EXPECTED_STORY_IDS[0] and r.base_seed == BASE_SEEDS[0]]
for record in pair:
    image = Image.open(RUN_DIR / record.comic_path)
    image.thumbnail((700, 700))
    print(f"{record.story_id} | nhánh {record.arm} | seed {record.base_seed}")
    display(image)